<a href="https://colab.research.google.com/github/SaulRodas/DeterministicWorkflowAgent/blob/main/EntropiaAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Technical Assessment**
---
**AI Engineer Intern | Entropia.ai**





*By: Saul Rodas*

#Setup

In [ ]:
!pip install langchain langchain-community langchain-core
!pip install langchain-google-genai
!pip install pandas
!pip install google-genai
!pip install faiss-cpu
!pip install sentence-transformers

In [ ]:
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
import os
from typing import List, Dict
import json
import re
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from PIL import Image
import base64
import mimetypes
from pydantic import BaseModel, Field

In [ ]:
from google.colab import userdata
api_key=userdata.get('api_key')

#LLM Connection

In [ ]:
chat_model = ChatGoogleGenerativeAI(
  model="gemini-3-flash-preview",
  google_api_key=api_key,
  temperature=0
)

#1.- Router

Le damos un diccionario con las fuentes de datos y una descripcion de que contiene, en caso de querer escalar a mas documentos

### Data Sources

In [ ]:
DATA_SOURCES = {
  "datos_clima_mexico.csv":
    "Dataset con temperaturas mensuales por estado en México, incluyendo meses, años y promedios climáticos.",
  "GPT-41_PromptingGuide.txt":
    "Guía técnica sobre prompting para GPT-4.1, incluyendo uso de etiquetas XML, structured prompting y buenas prácticas." ,
  "maiz_info.jpg":
    "Imagen con información visual sobre las razas de maíz nativas de México y su diversidad."
}

VALID_FILES = list(DATA_SOURCES.keys())

In [ ]:
#Transforma dict a txt plano
def format_data_sources(data_sources):
  formatted_text = ""
  for file, description in data_sources.items():
    formatted_text += f"{file} → {description}\n"

  return formatted_text

In [ ]:
sources_context = format_data_sources(DATA_SOURCES)
print(sources_context)

Usamos el LLM para que el router tome una decision

In [ ]:
#Prompt para el Router
router_system_prompt = f"""
Eres un Decision Maker encargado de seleccionar qué fuente de datos usar
para responder una pregunta del usuario.

Fuentes disponibles:

{sources_context}

Reglas:

- Responde SOLO con el nombre exacto del archivo.
- No expliques tu decisión.
- No agregues texto extra.
- No inventes archivos fuera de la lista.
"""

### Router

In [ ]:
def router_decision(user_question):
  messages = [
      SystemMessage(content=router_system_prompt),
      HumanMessage(content=user_question)
  ]

  response = chat_model.invoke(messages)

  decision = response.content[0]["text"].strip()

  # Validación básica
  if decision not in VALID_FILES:
      return "ERROR_INVALID_FILE"

  return decision

In [ ]:
decision=router_decision("Cual es el mes mas caliente de 2021?")
print(decision)

#2.- Retriever

## CSV

Para implementar nuestro retrieval en el CSV, vamos a hacer uso de un llm para identificar la intencion de la consulta y hacer un procesamiento del csv con pandas y asi poder darle un contexto a nuestro LLM de generacion.

In [ ]:
# Carga una copia del csv como dataframe con un formato mejor establecido de fechas
def load_csv(csv_path: str) -> pd.DataFrame:
  df = pd.read_csv(csv_path, sep=",")
  df["PERIODO"] = pd.to_datetime(df["PERIODO"])
  df["Año"] = df["PERIODO"].dt.year
  df["Mes"] = df["PERIODO"].dt.month
  return df

### LLM a Consulta

In [ ]:
# Prompt para identificar datos clave de la consulta que se quiere realizar

PARSER_SYSTEM_PROMPT = """
Eres un asistente que convierte preguntas en consultas estructuradas
para un dataset climático de México.

Reglas estrictas:
- NO respondas la pregunta.
- NO expliques nada.
- SOLO devuelve un JSON válido que cumpla el schema.
- Usa únicamente las operaciones permitidas.
- Si falta información, infiere solo lo explícito en la pregunta.

Operaciones permitidas:
- top_n_max_temperature
- top_n_min_temperature
- top_n_avg_temperature

Datos a devolver:
- Entidad
- Operacion
- Mes
- Año

No devuelvas extras
Meses deben devolverse como número (1-12).
"""


In [ ]:
#Convierte la salida cruda del LLM en un query limpio y usable.
def llm_output_to_query(llm_response: list) -> dict:
  raw_text = llm_response[0]["text"]
  parsed = json.loads(raw_text)
  query = {}

  for key, value in parsed.items():
    query[key] = value

  return query


### Retrieval

In [ ]:
# Funcion que de acuerdo a los datos del query obtenidos por el llm, hace la operacion al dataframe
def top_n_temperatures(df: pd.DataFrame, query: dict, top_n: int = 5):

  #Configuraciones para las 3 operaciones que se pueden obtener del dataframe(temp_maxima, temp_min, temp_average)
  OPERATION_CONFIG = {
    "top_n_max_temperature": {
      "column": "TEMP_MAXIMA",
      "ascending": False
    },
    "top_n_min_temperature": {
      "column": "TEMP_MINIMA",
      "ascending": True
    },
    "top_n_avg_temperature": {
      "column": "TEMP_MEDIA",
      "ascending": False
    }
  }

  #Identificamos operacion del dict query que devolvio llm
  operation = query.get("Operacion")

  if operation not in OPERATION_CONFIG:
    return {"error": f"Operación no soportada: {operation}"}

  config = OPERATION_CONFIG[operation]

  filtered = df.copy()

  # Filtros condicionales para el filtrado de los datos
  if query.get("Año") is not None:
    filtered = filtered[filtered["Año"] == query["Año"]]

  if query.get("Mes") is not None:
    filtered = filtered[filtered["Mes"] == query["Mes"]]

  if query.get("Entidad") is not None:
    filtered = filtered[filtered["ENTIDAD"] == query["Entidad"]]

  if filtered.empty:
    return []

  # Resultado del query al dataframe
  result = (
    filtered
    .sort_values(config["column"], ascending=config["ascending"])
    .head(top_n)
    [["ENTIDAD", config["column"], "Año", "Mes"]]
  )

  return result.to_dict(orient="records")


In [ ]:
#Funcion del Retrieval para CSV
def retrieve_from_csv(chat_model, data, user_question: str) -> str:

  #Cargamos .csv
  df=data

  #Cargamos System y Human messages

  messages = [
    SystemMessage(content=PARSER_SYSTEM_PROMPT),
    HumanMessage(content=user_question)]

  response = chat_model.invoke(messages) #Respuesta del LLM
  query = llm_output_to_query(response.content) #Query limpio
  print(query)

  context=top_n_temperatures(df, query) #Contexto para el LLM

  return context

In [ ]:
data=load_csv("/content/data/datos_clima_mexico.csv")
context=retrieve_from_csv(chat_model, data, "Cual es el mes mas caliente de 2020?")
print(context)

## TXT

In [ ]:
def load_txt(txt_path: str) -> str:
  try:
    with open(txt_path, "r", encoding="utf-8") as archivo:
      contenido = archivo.read()
      print("Archivo cargado")
      return contenido
  except FileNotFoundError:
    print("Error: El archivo no existe.")
    return None
  except PermissionError:
    print("Error: No tienes permisos para leer este archivo.")
    return None
  except Exception as e:
    print(f"Ocurrió un error: {e}")
    return None

### Chunking

Dado el formato del txt, dividido en temas y subtemas por markdowns y con bloques de codigo, lo mejor seria haer un chunking basado en esto. Para poder implementar nuestro RAG.

In [ ]:
def markdown_chunking(text, chunk_size=2000, overlap=200):
  lines = text.split('\n')
  chunks = []

  # Estado actual
  current_chunk = []
  current_size = 0
  in_code_block = False
  current_headers = {"h1": "", "h2": "", "h3": ""}

  # Regex para detectar encabezados markdown (# Título, ## Subtítulo)
  header_pattern = re.compile(r'^(#{1,3})\s+(.*)')

  for line in lines:
    # 1. Detectar si entramos o salimos de un bloque de código
    if line.strip().startswith('```'):
      in_code_block = not in_code_block
      current_chunk.append(line)
      current_size += len(line)
      continue

    # 2. Si NO estamos en un bloque de código, buscar encabezados
    match = header_pattern.match(line)
    if match and not in_code_block:
      level = len(match.group(1))
      title = match.group(2)

      # Si ya tenemos contenido acumulado, guardamos el chunk anterior
      if current_chunk:
        chunks.append({
          "content": "\n".join(current_chunk),
          "metadata": current_headers.copy() # Guardamos contexto del chunk anterior
        })
        current_chunk = []
        current_size = 0

      # Actualizamos la jerarquía de encabezados para el NUEVO chunk
      if level == 1:
        current_headers = {"h1": title, "h2": "", "h3": ""}
      elif level == 2:
        current_headers["h2"] = title
        current_headers["h3"] = ""
      elif level == 3:
        current_headers["h3"] = title

      # Opcional: Incluir el título en el contenido del texto también
      current_chunk.append(line)
      current_size += len(line)

    # 3. Procesamiento de líneas normales
    else:
      # Si el chunk es muy grande y hay un punto aparte, cortamos
      if current_size > chunk_size and not in_code_block and line.strip() == "":
        chunks.append({
          "content": "\n".join(current_chunk),
          "metadata": current_headers.copy()
        })
        # Mantenemos un overlap (solapamiento) simple tomando las últimas líneas
        overlap_text = current_chunk[-5:] if len(current_chunk) > 5 else []
        current_chunk = list(overlap_text)
        current_size = sum(len(l) for l in current_chunk)
      else:
        current_chunk.append(line)
        current_size += len(line)

  # Agregar el último chunk pendiente
  if current_chunk:
    chunks.append({
      "content": "\n".join(current_chunk),
      "metadata": current_headers.copy()
    })

  return chunks

In [ ]:
text_content = load_txt("/content/data/GPT-41_PromptingGuide.txt")
chunks = markdown_chunking(text_content)

print(f"Total de chunks generados: {len(chunks)}")
print(chunks[0])

### Embedding Chunks

In [ ]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")

def embed_chunks(chunks):
  texts = []
  metadatas = []

  for c in chunks:
    text = f"""
    {c['metadata']['h1']}
    {c['metadata']['h2']}
    {c['metadata']['h3']}

    {c['content']}
    """
    texts.append(text.strip())
    metadatas.append(c['metadata']) # This line was added


  vectors = embedding_model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
  )

  return texts, vectors, metadatas

### FAISS

In [ ]:
def build_faiss_index(vectors):
  dim = vectors.shape[1]
  index = faiss.IndexFlatIP(dim)
  index.add(vectors.astype("float32"))
  return index

In [ ]:
texts, vectors, metadatas = embed_chunks(chunks)
index = build_faiss_index(vectors)

### Retrieval

In [ ]:
def rag_txt(txt_path):
  text_content = load_txt(txt_path)
  chunks = markdown_chunking(text_content)

  texts, vectors, metadatas = embed_chunks(chunks)
  index = build_faiss_index(vectors)

  return texts, index, metadatas

In [ ]:
def retrieve_txt(query, index, texts, metadatas, k=1):
  query_vector = embedding_model.encode(
    [query],
    normalize_embeddings=True
  )

  scores, indices = index.search(
    query_vector.astype("float32"), k
  )

  context = []
  for idx in indices[0]:
    context.append({
      "content": texts[idx],
      "metadata": metadatas[idx]
    })

  return context

In [ ]:
context=retrieve_txt("¿Qué dice el cookbook sobre el uso de etiquetas XML?", index, texts, metadatas)
print(context)

## IMG
Al usar un modelo LLM multimodal, simplemente le pasaremos la imagen, en caso de no tener alguno, se tendrian que emplear tecnicas de OCR

In [ ]:
def retrieve_img(img_path):
    mime_type, _ = mimetypes.guess_type(img_path)
    if not mime_type:
        mime_type = "image/jpeg"

    with open(img_path, "rb") as f:
        encoded_string = base64.b64encode(f.read()).decode("utf-8")

    return {
        "type": "image_url",
        "image_url": {"url": f"data:{mime_type};base64,{encoded_string}"}
    }

In [ ]:
image_path = "/content/data/maiz_info.jpg"
context=retrieve_img(image_path)
print(context)

## Retriever Function

### Prompts dinamicos

In [ ]:
img_prompt= """
Actúa como un observador visual de alta precisión.
Analizarás la imagen proporcionada para responder a la consulta del usuario.

Reglas:
- Describe o responde basándote solo en lo que es claramente visible en la imagen.
- Si el usuario pregunta por un detalle que no aparece, que está fuera de cuadro o que es ilegible (texto borroso, objetos oscuros), responde: 'La imagen no proporciona información suficiente sobre [detalle específico]'.
- No realices suposiciones sobre lo que podría haber sucedido antes o después de la captura, ni sobre elementos que no están explícitamente presentes.
- Si se te pide identificar un objeto y no hay certeza visual absoluta, describe sus características físicas en lugar de darle un nombre definitivo.
"""

In [ ]:
txt_prompt= """
Actúa como un especialista en extracción de información. Se te proporcionará un texto como contexto único para responder a las dudas del usuario.

Reglas:
- Tu respuesta debe provenir únicamente del texto suministrado.
- Si la respuesta no está presente de forma explícita o implícita directa en el texto, debes decir: 'No puedo responder a esto porque la información no está disponible en el texto de referencia'.
- No añadas información adicional, aunque sepas que es cierta en el mundo real.
- Mantén una postura neutral y objetiva.
"""

In [ ]:
csv_prompt="""
Actúa como un analista de datos riguroso. Tu única fuente de información son los datos filtrados de un CSV que te proporcionan.

Reglas:
- Responde preguntas basándote exclusivamente en las filas y columnas proporcionadas.
- Si la consulta requiere comparar datos que no están presentes o si la respuesta no se puede deducir de los datos suministrados, responde exactamente: 'La información solicitada no se encuentra en los registros proporcionados'.
- No utilices conocimiento externo ni inventes tendencias.
- Si los datos están incompletos para realizar un cálculo, indícalo claramente.
"""

### Build Messages

In [ ]:
def build_messages(router_output: str, user_question: str):

    if router_output.endswith(".txt"):

        context = retrieve_txt(user_question, index, texts, metadatas)
        content_system=txt_prompt

        # Extract and join the 'content' from each dictionary in the context list
        context_str = "\n\n".join([item["content"] for item in context])

        content_human =[
          {"type": "text", "text": user_question},
          {"type": "text", "text": context_str}
        ]

    elif router_output.endswith(".csv"):
        context = retrieve_from_csv(chat_model, data, user_question)
        content_system=csv_prompt

        # Convert the list of dictionaries to a string representation
        context_str = str(context)

        content_human =[
          {"type": "text", "text": user_question},
          {"type": "text", "text": context_str}
        ]

    elif router_output.endswith(".jpg"):
        context = retrieve_img(image_path)
        content_system=img_prompt

        content_human =[
          {"type": "text", "text": user_question},
          context
        ]

    else:
        raise ValueError(f"Tipo de archivo no soportado: {router_output}")

    return content_system, content_human

In [ ]:
m1, m2 = build_messages("maiz_info.jpg", "¿De que trata la imagen?")
print(m1)
print(m2)

### Response LLM

In [ ]:
def raw_response_llm(content_system, content_human):
  messages = [
    SystemMessage(content=content_system),
    HumanMessage(content=content_human)
  ]

  response = chat_model.invoke(messages)
  return response.content[0]['text']

In [ ]:
response=raw_response_llm(m1, m2)
print(response)

#3.- Stylist

In [ ]:
class StyledAnswer(BaseModel):
  """Esquema para la salida estructurada del Stylist."""
  original_answer: str = Field(description="La respuesta original generada en el paso anterior.")
  dry_answer: str = Field(description="Una versión muy concisa, directa y profesional.")
  funny_answer: str = Field(description="Una versión con humor y mucha personalidad.")

In [ ]:
Stilyst_system_prompt = """
Eres un editor de estilo experto ("The Stylist").
Tu tarea es recibir información cruda y transformarla en tres formatos distintos según el esquema solicitado.

1. Original: Mantén exactamente el mismo texto crudo, no hagas ningun cambio o variacion.
2. Dry: Sé extremadamente conciso, elimina adornos, ve al grano (estilo corporativo/científico).
3. Funny: Usa jerga mexicana, emojis y un tono divertido/sarcástico, pero mantén la veracidad de los datos.
"""

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def stylist_agent(raw_context: str):

    structured_llm = chat_model.with_structured_output(StyledAnswer)

    prompt = ChatPromptTemplate.from_messages([
        ("system", """
          Eres un editor de estilo experto ("The Stylist").
          Tu tarea es recibir información cruda y transformarla en tres formatos distintos según el esquema solicitado.

          1. Original: Mantén exactamente el mismo texto crudo, no hagas ningun cambio o variacion.
          2. Dry: Sé extremadamente conciso, elimina adornos, ve al grano (estilo corporativo/científico).
          3. Funny: Usa jerga mexicana, emojis y un tono divertido/sarcástico, pero mantén la veracidad de los datos.
          """),
        ("human", "Aquí está la información cruda para procesar: {input_text}")
    ])

    chain = prompt | structured_llm

    return chain.invoke({"input_text": raw_context})

In [ ]:
# Llamamos a la función
resultado_final = stylist_agent(response)

# Imprimimos el resultado (que será un objeto StyledAnswer)
print("--- JSON FINAL ---")
print(resultado_final.model_dump_json(indent=2))

#Workflow

Carga archivos

In [ ]:
DATA_SOURCES = {
  "datos_clima_mexico.csv":
    "Dataset con temperaturas mensuales por estado en México, incluyendo meses, años y promedios climáticos.",
  "GPT-41_PromptingGuide.txt":
    "Guía técnica sobre prompting para GPT-4.1, incluyendo uso de etiquetas XML, structured prompting y buenas prácticas." ,
  "maiz_info.jpg":
    "Imagen con información visual sobre las razas de maíz nativas de México y su diversidad."
}

VALID_FILES = list(DATA_SOURCES.keys())
sources_context = format_data_sources(DATA_SOURCES)

In [ ]:
image_path="/content/data/maiz_info.jpg"
txt_path="/content/data/GPT-41_PromptingGuide.txt"
csv_path="/content/data/datos_clima_mexico.csv"

In [ ]:
data=load_csv(csv_path)
data.head(5)

In [ ]:
texts, index, metadatas = rag_txt(txt_path)

In [ ]:
def deterministic_workflow(user_question: str):

  #1. Router
  router_output = router_decision(user_question)

  #2. Retriever
  content_system, content_human = build_messages(router_output, user_question)

  #3. Stylist
  raw_context = raw_response_llm(content_system, content_human)
  stylist_output = stylist_agent(raw_context)

  return stylist_output


## Pruebas

In [ ]:
p1= deterministic_workflow("¿Qué dice el cookbook sobre el uso de etiquetas XML?")
print(p1)

In [ ]:
p2= deterministic_workflow("¿Algún consejo para promptear a GPT-5?")
print(p2)

In [ ]:
p3= deterministic_workflow("¿Top 5 estados más calientes en agosto de 2021?")
print(p3)

In [ ]:
p4= deterministic_workflow("¿Cuáles fueron las 3 temperaturas mínimas en diciembre de 2025?")
print(p4)

In [ ]:
p5= deterministic_workflow("¿Cuántas razas de maíz son nativas en México?")
print(p5)

In [ ]:
p6= deterministic_workflow("¿Por qué el huitlacoche es superior en nutrientes al maíz tradicional?")
print(p6)

# 8. Final Reflection

#### Tokens & Cost
**Is this architecture scalable? How to optimize?**

*It is scalable for low/medium volumes, currently it uses several calls to LLM for each question, so it has a linear cost.*

*Perhaps to optimize this we could make more compact chunks, try to avoid using LLM, for example to parse the queries to the CSV, looking for other methods for this, and avoid steps like the stylizer.*


---


#### Latency
**Is it too slow for real-time chat?**

*For real-time chat, it may feel slow due to chaining multiple synchronous steps.*

*To improve this, we could try using faster models for tasks like the router/parser and let the strong model handle only the final response.*


---



### Evaluation
**How can we prove accuracy beyond “eye-balling”?**

*“Eye-balling” is not enough, we need a reproducible benchmark. Maybe build a dataset of questions with golden answers and the expected source file (txt/csv/jpg) or define metrics per stage: router accuracy, retrieval recall@k, and groundedness/factuality.*



---



### Autonomy
**What if we used a ReAct Agent instead? Pros/Cons?**

The difference lies in flow control: in ReAct, the LLM dynamically decides the next step, whereas in our deterministic workflow the path is predefined (router → retriever → prompt → generator → stylist), even if we use LLMs for specific tasks like query parsing. This makes ReAct more flexible for open-ended problems, but also more costly, while the deterministic approach prioritizes reliability, low latency, and ease of testing.


*ReAct — Pros*
- More flexible for ambiguous or multi-hop tasks.
- Better exploration when the solution path is not known in advance.

*ReAct — Cons*
- Higher cost and latency due to extra turns and tool calls.
- Lower traceability and higher deviation risk.
- Harder to test and control in production.


---



### Personal Retrospective
**What was your biggest takeaway? Did you enjoy the struggle? Do you see yourself building these solutions daily?**

My biggest takeaway was around context engineering; I had never designed anything like that before. Understanding how to structure, filter, and prioritize context so the model can respond accurately. It made me realize that it’s not just about the model, but about what information you give it, how you provide it, and at what point in the pipeline.

I did enjoy the struggle, since it forced me to think carefully about the flow and how the internal connections between the functions I built should work, as well as how their outputs would impact the next stages of the pipeline. And yes, I do see myself building these kinds of solutions.
